## 0. Setup

**Dataset:** NYC Yellow Taxi Trip Records — Jan 2022 + Jan 2023 (~6M rows combined)

| API | Abstraction | Lazy? | Distributed? | Typed schema? |
|-----|-------------|-------|--------------|---------------|
| RDD | Low-level (Python objects) | Yes | Yes | No |
| Pandas DataFrame | High-level | No (eager) | No (single node) | Yes (dtypes) |
| Spark DataFrame | High-level | Yes | Yes | Yes (StructType) |


In [ ]:
# Install dependencies if needed
# !pip install pyspark pandas pyarrow matplotlib requests

In [1]:
import os
import math
import time
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
spark = (
    SparkSession.builder
    .appName("PySpark_Demo_NYC_Taxi")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "2")
    .config("spark.sql.shuffle.partitions", "8")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# FileStreamSink.hasMetadata() always warns on glob paths in Spark 4.x — silence it
log4j = spark.sparkContext._jvm.org.apache.log4j
log4j.Logger.getLogger(
    "org.apache.spark.sql.execution.streaming.sinks.FileStreamSink"
).setLevel(log4j.Level.ERROR)

sc = spark.sparkContext

print(f"Spark version      : {spark.version}")
print(f"Master             : {spark.sparkContext.master}")
print(f"Parallel workers   : {sc.defaultParallelism}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/22 17:44:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version      : 4.1.1
Master             : local[*]
Parallel workers   : 8


---
## 1. Download Both Years of NYC Taxi Data

We download **Jan 2022** and **Jan 2023** parquet files (~50 MB each, ~3M rows each).
Combining them gives us ~6M rows — enough to start seeing Spark's multi-file advantages.

In [3]:
DATA_DIR = os.path.abspath(os.getcwd())  # absolute path suppresses FileStreamSink WARN

FILES = {
    "yellow_tripdata_2021-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2021-01.parquet",
    "yellow_tripdata_2022-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2022-01.parquet",
    "yellow_tripdata_2023-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet",
    "yellow_tripdata_2024-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
    "yellow_tripdata_2025-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet",
}

for fname, url in FILES.items():
    fpath = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fpath):
        print(f"Downloading {fname} ...")
        r = requests.get(url, stream=True)
        r.raise_for_status()
        with open(fpath, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"  Saved ({os.path.getsize(fpath)/1e6:.1f} MB)")
    else:
        print(f"  Found cached: {fname} ({os.path.getsize(fpath)/1e6:.1f} MB)")

DATA_2021 = os.path.join(DATA_DIR, "yellow_tripdata_2021-01.parquet")
DATA_2022 = os.path.join(DATA_DIR, "yellow_tripdata_2022-01.parquet")
DATA_2023 = os.path.join(DATA_DIR, "yellow_tripdata_2023-01.parquet")
DATA_2024 = os.path.join(DATA_DIR, "yellow_tripdata_2024-01.parquet")
DATA_2025 = os.path.join(DATA_DIR, "yellow_tripdata_2025-01.parquet")
DATA_GLOB = os.path.join(DATA_DIR, "yellow_tripdata_202*.parquet")  # absolute glob


  Found cached: yellow_tripdata_2021-01.parquet (21.7 MB)
  Found cached: yellow_tripdata_2022-01.parquet (38.1 MB)
  Found cached: yellow_tripdata_2023-01.parquet (47.7 MB)
  Found cached: yellow_tripdata_2024-01.parquet (50.0 MB)
  Found cached: yellow_tripdata_2025-01.parquet (59.2 MB)


---
## 2. Combining Files Using Spark Dataframe


In [4]:
# ── SPARK DF: read files in parallel, handling schema drift across years ────
# Note: 2024+ files store passenger_count as INT64 vs DoubleType in 2021-2023.
# Reading each file with its own schema then union avoids the vectorized-reader
# type-mismatch error (PARQUET_COLUMN_DATA_TYPE_MISMATCH).
print("[Spark DF] Reading all years with schema-safe union...")

t0 = time.perf_counter()

def _read_year(path):
    df = spark.read.parquet(path)
    # Normalize passenger_count to LongType — 2024+ changed it from DoubleType
    return df.withColumn("passenger_count", F.col("passenger_count").cast("long"))

all_files = [DATA_2021, DATA_2022, DATA_2023, DATA_2024, DATA_2025]
sdf = _read_year(all_files[0])
for _path in all_files[1:]:
    sdf = sdf.unionByName(_read_year(_path), allowMissingColumns=True)

plan_time = time.perf_counter() - t0
print(f"  Plan built in {plan_time*1000:.1f} ms  ← lazy, no I/O yet")

t0 = time.perf_counter()
sdf_count = sdf.count()
t_spark_combine = time.perf_counter() - t0

print(f"  Combined  : {sdf_count:,} rows")
print(f"  Time      : {t_spark_combine:.2f}s  (parallel: all files read simultaneously)")
print(f"  Partitions: {sdf.rdd.getNumPartitions()}")
sdf.show(5, truncate=False)

[Spark DF] Reading all years with schema-safe union...


  Plan built in 5407.5 ms  ← lazy, no I/O yet


  Combined  : 13,340,316 rows
  Time      : 3.03s  (parallel: all files read simultaneously)
  Partitions: 38
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|1       |2021-01-01

In [5]:
# ── Schema alignment check ────────────────────────────────────────────────────
# Real-world concern: schemas can drift between years (columns added/renamed/retyped)
print("Checking for schema differences between 2022 and 2023 ...")

schema_2022 = {f.name: str(f.dataType) for f in spark.read.parquet(DATA_2022).schema.fields}
schema_2023 = {f.name: str(f.dataType) for f in spark.read.parquet(DATA_2023).schema.fields}

only_in_2022 = set(schema_2022) - set(schema_2023)
only_in_2023 = set(schema_2023) - set(schema_2022)
type_changed = {c for c in set(schema_2022) & set(schema_2023) if schema_2022[c] != schema_2023[c]}

if not any([only_in_2022, only_in_2023, type_changed]):
    print("  Schemas match perfectly — safe to concat.")
else:
    if only_in_2022:  print(f"  Only in 2022 : {only_in_2022}")
    if only_in_2023:  print(f"  Only in 2023 : {only_in_2023}")
    if type_changed:  print(f"  Type changed : {type_changed}")
    print("  Note: Spark handles schema evolution automatically with mergeSchema option.")

print(f"\nTotal columns: {len(schema_2023)}")
sdf.printSchema()

Checking for schema differences between 2022 and 2023 ...
  Schemas match perfectly — safe to concat.

Total columns: 19
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_con

---
## 14. Section 8 — Distributed Processing Concepts

```
Driver                Executors (workers)
  │                       │   │   │
  │──── tasks ───────────>│   │   │
  │<─── results ──────────│   │   │
  │
  └── manages: DAG, scheduling, fault tolerance
```

| Concept | Why it matters |
|---------|---------------|
| Partitions | Unit of parallelism — each partition = one task on one core |
| Shuffle | Most expensive operation — data moves across the network |
| Skew | One partition with 10× more data than others → bottleneck |
| `repartition` | Full shuffle, even distribution — use to increase partitions or hash by column |
| `coalesce` | Reduces partitions **without** shuffle — use before writing |
| Broadcast join | Small table sent to all workers — eliminates shuffle on the large table |

In [5]:
# --- Partitioning ---
print("=== Partitioning ===")
print(f"sdf partitions     : {sdf.rdd.getNumPartitions()}")
print(f"shuffle.partitions : {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"default parallelism: {sc.defaultParallelism}\n")

# Inspect row distribution across partitions
part_dist = (
    sdf.withColumn("part", F.spark_partition_id())
    .groupBy("part").count()
    .orderBy("part")
)
print("Row count per partition (current):")
part_dist.show()

# --- repartition vs coalesce ---
print("=== repartition() vs coalesce() ===")

t0 = time.perf_counter()
sdf_rep = sdf.repartition(16)
_ = sdf_rep.count()
t_rep = time.perf_counter() - t0
print(f"repartition(16) → {sdf_rep.rdd.getNumPartitions()} partitions, {t_rep:.2f}s  ← FULL SHUFFLE")

t0 = time.perf_counter()
sdf_coal = sdf.coalesce(2)
_ = sdf_coal.count()
t_coal = time.perf_counter() - t0
print(f"coalesce(2)     → {sdf_coal.rdd.getNumPartitions()} partitions, {t_coal:.2f}s  ← no shuffle")

# Hash-partition by a key (co-locate related rows)
sdf_hash = sdf.repartition(8, "PULocationID")
print(f"\nrepartition(8, 'PULocationID'): trips with same zone → same partition")
(sdf_hash.withColumn("part", F.spark_partition_id())
 .groupBy("part","PULocationID").count()
 .orderBy("part","PULocationID")
 .filter(F.col("PULocationID").isin([132,161,236]))
 .show(10))

print("""
Use repartition when: increasing partitions, changing partition key, evening skew
Use coalesce when   : reducing partitions before a write (avoids extra shuffle)
""")

=== Partitioning ===
sdf partitions     : 38
shuffle.partitions : 8
default parallelism: 8

Row count per partition (current):


+----+-------+
|part|  count|
+----+-------+
|   2|1369769|
|   9|2463931|
|  17|3066766|
|  23|1048576|
|  25|1048576|
|  28| 867472|
|  31|1048576|
|  33|1048576|
|  35|1048576|
|  37| 329498|
+----+-------+

=== repartition() vs coalesce() ===


repartition(16) → 16 partitions, 3.39s  ← FULL SHUFFLE


coalesce(2)     → 2 partitions, 1.03s  ← no shuffle

repartition(8, 'PULocationID'): trips with same zone → same partition


+----+------------+------+
|part|PULocationID| count|
+----+------------+------+
|   3|         236|625714|
|   4|         161|581024|
|   6|         132|587263|
+----+------------+------+


Use repartition when: increasing partitions, changing partition key, evening skew
Use coalesce when   : reducing partitions before a write (avoids extra shuffle)



In [6]:
# --- Shuffle and Skew ---
print("=== Shuffle: what happens during groupBy / join ===")
print("""
1. MAP phase   : each task hashes each row's key → assigns to target partition number
2. SHUFFLE     : rows move across the network to their target partition
3. REDUCE phase: each partition aggregates/joins its assigned rows

Shuffle writes intermediate data to disk → most expensive Spark operation.
Minimize shuffles:
  ✓ Broadcast small tables (< threshold) to avoid shuffle on the large side
  ✓ Pre-partition data on join key before multiple joins
  ✓ Increase spark.sql.shuffle.partitions for large shuffles
""")

print("=== Data Skew Demo ===")
# Simulate skewed data: key=1 has 100× more rows than others
skewed = (
    [(1, float(i)) for i in range(100_000)] +
    [(2, float(i)) for i in range(1_000)]  +
    [(3, float(i)) for i in range(500)]
)
sdf_skew = spark.createDataFrame(skewed, ["key","value"]).repartition(4, "key")
#sdf_skew = spark.createDataFrame(skewed, ["key","value"])

print("Distribution with skew (key=1 dominates):")
(sdf_skew.withColumn("part", F.spark_partition_id())
 .groupBy("part","key").count()
 .orderBy("part","key").show())

# Fix: salting — spread key=1 across multiple partitions
SALT = 4
sdf_salted = sdf_skew.withColumn(
    "salted_key",
    F.concat(F.col("key").cast("string"), F.lit("_"),
             (F.rand(seed=42) * SALT).cast("int").cast("string"))    # create 4 salted versions of key=1, 2,3
).repartition(8, "salted_key")

print("Distribution after salting (key=1 spread across partitions):")
(sdf_salted.withColumn("part", F.spark_partition_id())
 .groupBy("part","key").count()
 .orderBy("part","key").show())

(sdf_salted
 .withColumn("part", F.spark_partition_id())
 .groupBy("part", "key", "salted_key")
 .count()
 .orderBy("key", "salted_key", "part")
 .show(100, truncate=False))




=== Shuffle: what happens during groupBy / join ===

1. MAP phase   : each task hashes each row's key → assigns to target partition number
2. SHUFFLE     : rows move across the network to their target partition
3. REDUCE phase: each partition aggregates/joins its assigned rows

Shuffle writes intermediate data to disk → most expensive Spark operation.
Minimize shuffles:
  ✓ Broadcast small tables (< threshold) to avoid shuffle on the large side
  ✓ Pre-partition data on join key before multiple joins
  ✓ Increase spark.sql.shuffle.partitions for large shuffles

=== Data Skew Demo ===
Distribution with skew (key=1 dominates):


+----+---+------+
|part|key| count|
+----+---+------+
|   0|  2|  1000|
|   1|  1|100000|
|   3|  3|   500|
+----+---+------+

Distribution after salting (key=1 spread across partitions):
+----+---+-----+
|part|key|count|
+----+---+-----+
|   0|  1|24969|
|   0|  2|  254|
|   0|  3|  114|
|   2|  1|24928|
|   2|  3|  121|
|   3|  3|  122|
|   5|  2|  538|
|   6|  2|  208|
|   6|  3|  143|
|   7|  1|50103|
+----+---+-----+

+----+---+----------+-----+
|part|key|salted_key|count|
+----+---+----------+-----+
|7   |1  |1_0       |25190|
|0   |1  |1_1       |24969|
|2   |1  |1_2       |24928|
|7   |1  |1_3       |24913|
|0   |2  |2_0       |254  |
|5   |2  |2_1       |264  |
|5   |2  |2_2       |274  |
|6   |2  |2_3       |208  |
|6   |3  |3_0       |143  |
|2   |3  |3_1       |121  |
|3   |3  |3_2       |122  |
|0   |3  |3_3       |114  |
+----+---+----------+-----+



In [7]:
# --- Broadcast Joins ---
from pyspark.sql.functions import broadcast

# --- Joins ---
# Small zone lookup table
zone_data = [
    (132, "JFK Airport"),         (138, "LaGuardia Airport"),
    (161, "Midtown Center"),      (236, "Upper East Side N"),
    (237, "Upper East Side S"),   (142, "Lincoln Square E"),
    (230, "Times Sq/Theatre"),    (170, "Murray Hill"),
]
zone_df = spark.createDataFrame(zone_data, ["LocationID", "Zone"])

print("=== Broadcast Joins ===")
print(f"Auto-broadcast threshold: {spark.conf.get('spark.sql.autoBroadcastJoinThreshold')} bytes (default 10 MB)")
print("zone_df has only 8 rows — ideal broadcast candidate.\n")

# Shuffle join (Spark may auto-broadcast since zone_df is tiny, so we disable)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # force shuffle join
t0 = time.perf_counter()
n1 = sdf.join(zone_df, sdf["PULocationID"] == zone_df["LocationID"], "left").count()
t_shuffle = time.perf_counter() - t0

# Restore and use explicit broadcast
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
t0 = time.perf_counter()
n2 = sdf.join(broadcast(zone_df), sdf["PULocationID"] == zone_df["LocationID"], "left").count()
t_broadcast = time.perf_counter() - t0

print(f"SortMerge join   : {t_shuffle:.3f}s")
print(f"Broadcast join   : {t_broadcast:.3f}s  ({t_shuffle/max(t_broadcast,0.001):.1f}× faster)")

print("\n--- Verify BroadcastHashJoin in plan ---")
sdf.join(broadcast(zone_df), sdf["PULocationID"] == zone_df["LocationID"], "left").explain(mode="simple")

print("""
Rule of thumb:
  Broadcast the smaller table when it fits in driver + executor memory.
  Default threshold: 10 MB.
  Force: broadcast(df)  OR  spark.conf.set('spark.sql.autoBroadcastJoinThreshold', bytes)
""")

=== Broadcast Joins ===
Auto-broadcast threshold: 10485760b bytes (default 10 MB)
zone_df has only 8 rows — ideal broadcast candidate.



SortMerge join   : 5.104s
Broadcast join   : 1.130s  (4.5× faster)

--- Verify BroadcastHashJoin in plan ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [PULocationID#7L], [LocationID#486L], LeftOuter, BuildRight, false
   :- Union
   :  :- Project [VendorID#0L, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, cast(passenger_count#3 as bigint) AS passenger_count#20L, trip_distance#4, RatecodeID#5, store_and_fwd_flag#6, PULocationID#7L, DOLocationID#8L, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, airport_fee#18, null AS cbd_congestion_fee#110]
   :  :  +- FileScan parquet [VendorID#0L,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3,trip_distance#4,RatecodeID#5,store_and_fwd_flag#6,PULocationID#7L,DOLocationID#8L,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount

---
## 15. Section 9 — Data Storage Formats

| Format | Row/Col | Schema | Splittable | Compression | Best for |
|--------|---------|--------|-----------|------------|---------|
| CSV | Row | ✗ | ✓ (with index) | ✗ | Human-readable exchange |
| JSON | Row | Partial | ✓ | ✗ | Semi-structured / APIs |
| **Parquet** | **Columnar** | **✓** | **✓** | **✓** | **Analytics — default choice** |
| ORC | Columnar | ✓ | ✓ | ✓ | Hive / HBase workloads |

**Why columnar?** A query reading only 3 of 19 columns skips 16 columns entirely — reads ~16% of the bytes. Row formats must scan every column regardless.

In [8]:
print("=== Write Same Data in 4 Formats, Compare File Sizes ===\n")

# 500K rows sample
sdf_fmt = sdf.filter(F.year("tpep_pickup_datetime") == 2023).limit(500_000)

fmt_paths = {
    "csv":     f"{DATA_DIR}/fmt_csv",
    "json":    f"{DATA_DIR}/fmt_json",
    "parquet": f"{DATA_DIR}/fmt_parquet",
    "orc":     f"{DATA_DIR}/fmt_orc",
}

write_times, read_times, sizes = {}, {}, {}

for fmt, path in fmt_paths.items():
    if os.path.exists(path): shutil.rmtree(path)
    t0 = time.perf_counter()
    w = sdf_fmt.write.mode("overwrite")
    if fmt == "csv":     w.option("header", True).csv(path)
    elif fmt == "json":  w.json(path)
    elif fmt == "parquet": w.parquet(path)
    elif fmt == "orc":   w.orc(path)
    write_times[fmt] = time.perf_counter() - t0
    sizes[fmt] = sum(os.path.getsize(os.path.join(dp, f))
                     for dp, _, fns in os.walk(path) for f in fns) / 1e6

print(f"{'Format':10s} {'Size (MB)':>10} {'Write (s)':>10}")
print("-" * 32)
for fmt in fmt_paths:
    print(f"{fmt:10s} {sizes[fmt]:>10.1f} {write_times[fmt]:>10.2f}")

print(f"\nParquet is {sizes['csv']/sizes['parquet']:.1f}× smaller than CSV")
print(f"Parquet is {sizes['json']/sizes['parquet']:.1f}× smaller than JSON")

=== Write Same Data in 4 Formats, Compare File Sizes ===



Format      Size (MB)  Write (s)
--------------------------------
csv              56.6       5.37
json            214.8       4.68
parquet          10.6       4.48
orc               9.7       4.07

Parquet is 5.3× smaller than CSV
Parquet is 20.3× smaller than JSON


In [9]:
print("=== Read Performance by Format (full scan, count) ===\n")
for fmt, path in fmt_paths.items():
    t0 = time.perf_counter()
    if fmt == "csv":     n = spark.read.option("header",True).option("inferSchema",True).csv(path).count()
    elif fmt == "json":  n = spark.read.json(path).count()
    elif fmt == "parquet": n = spark.read.parquet(path).count()
    elif fmt == "orc":   n = spark.read.orc(path).count()
    read_times[fmt] = time.perf_counter() - t0
    print(f"{fmt:10s}: {n:,} rows in {read_times[fmt]:.2f}s")

print("\n=== Column Pruning: Parquet reads ONLY requested columns ===")
t0 = time.perf_counter()
spark.read.parquet(fmt_paths["parquet"]).select("fare_amount","trip_distance").count()
t_pruned = time.perf_counter() - t0

t0 = time.perf_counter()
spark.read.parquet(fmt_paths["parquet"]).count()
t_full = time.perf_counter() - t0

print(f"Read 2/19 columns  : {t_pruned:.3f}s")
print(f"Read all 19 columns: {t_full:.3f}s")
print(f"Column pruning speedup: {t_full/max(t_pruned,0.001):.1f}×\n")

print("=== Parquet Compression Comparison ===")
for comp in ["snappy", "gzip", "zstd", "lz4"]:
    path = f"{DATA_DIR}/parquet_{comp}"
    if os.path.exists(path): shutil.rmtree(path)
    t0 = time.perf_counter()
    sdf_fmt.write.mode("overwrite").option("compression", comp).parquet(path)
    tw = time.perf_counter() - t0
    sz = sum(os.path.getsize(os.path.join(dp,f)) for dp,_,fns in os.walk(path) for f in fns)/1e6
    t0 = time.perf_counter()
    spark.read.parquet(path).count()
    tr = time.perf_counter() - t0
    print(f"{comp:8s}: {sz:5.1f} MB  write:{tw:.2f}s  read:{tr:.2f}s")

print("\nsnappy=default(fast decompress), zstd=best balance, gzip=smallest(cold archive)")

=== Read Performance by Format (full scan, count) ===



csv       : 500,000 rows in 2.91s


json      : 500,000 rows in 2.60s
parquet   : 500,000 rows in 0.30s
orc       : 500,000 rows in 0.30s

=== Column Pruning: Parquet reads ONLY requested columns ===
Read 2/19 columns  : 0.214s
Read all 19 columns: 0.283s
Column pruning speedup: 1.3×

=== Parquet Compression Comparison ===


snappy  :  10.6 MB  write:5.01s  read:0.17s


gzip    :   8.3 MB  write:4.27s  read:0.14s


zstd    :   8.1 MB  write:3.69s  read:0.15s


lz4     :  10.5 MB  write:3.56s  read:0.15s

snappy=default(fast decompress), zstd=best balance, gzip=smallest(cold archive)


---
## 16. Section 10 — ETL / ELT Pipeline Design

```
ETL:  Extract → Transform → Load   (transform before landing)
ELT:  Extract → Load → Transform   (land raw, transform in warehouse/lake)
```

Key engineering properties:

| Property | Description |
|----------|------------|
| **Idempotency** | Running the pipeline twice produces the same result |
| **Incremental** | Process only new/changed records, not full reload |
| **Fault tolerance** | Failures retry without corrupting state |
| **Observability** | Logs and metrics for debugging and monitoring |
| **Partitioning** | Output partitioned by date/key for efficient downstream reads |

In [11]:
import logging, traceback

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("taxi_etl")

# ── ETL functions ──────────────────────────────────────────────────────────────

def extract(spark, source_path):
    logger.info(f"EXTRACT from {source_path}")
    df = spark.read.parquet(source_path)
    logger.info(f"  extracted {df.count():,} rows, {len(df.columns)} columns")
    return df

def validate_schema(df, required_cols, name="data"):
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{name}] Missing required columns: {missing}")
    logger.info(f"  schema OK — {len(required_cols)} required columns present")

def transform(df):
    return (
        df
        .filter(F.col("fare_amount").between(0.5, 1000))
        .filter(F.col("trip_distance").between(0, 500))
        .filter(F.col("tpep_pickup_datetime").isNotNull())
        .filter(F.col("tpep_pickup_datetime") < F.col("tpep_dropoff_datetime"))
        .fillna({"passenger_count": 1, "airport_fee": 0.0, "congestion_surcharge": 0.0})
        .dropDuplicates(["tpep_pickup_datetime","PULocationID","DOLocationID","fare_amount"])
        .withColumn("year",         F.year("tpep_pickup_datetime"))
        .withColumn("month",        F.month("tpep_pickup_datetime"))
        .withColumn("pickup_hour",  F.hour("tpep_pickup_datetime"))
        .withColumn("duration_min", F.round(
            (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60, 1
        ))
        .withColumn("fare_band",
            F.when(F.col("fare_amount") < 10, "low")
             .when(F.col("fare_amount") < 25, "medium")
             .otherwise("high")
        )
    )

def load(df, output_path, partition_cols=None):
    writer = df.write.mode("overwrite")
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.parquet(output_path)
    logger.info(f"  loaded → {output_path}")

# ── Run full batch pipeline ────────────────────────────────────────────────────
ETL_OUT = f"{DATA_DIR}/etl_output"
if os.path.exists(ETL_OUT): shutil.rmtree(ETL_OUT)

try:
    t0 = time.perf_counter()
    raw = extract(spark, DATA_2023)
    validate_schema(raw, ["fare_amount","tpep_pickup_datetime","trip_distance"])
    clean = transform(raw)
    load(clean, ETL_OUT, partition_cols=["year","month"])
    logger.info(f"Pipeline complete in {time.perf_counter()-t0:.2f}s")
except Exception as e:
    logger.error(f"Pipeline failed: {e}\n{traceback.format_exc()}")
    raise

print(f"\nOutput directories (partitioned by year/month):")
for d in sorted(os.listdir(ETL_OUT)):
    if os.path.isdir(os.path.join(ETL_OUT, d)): print(f"  {d}/")

2026-05-21 16:14:12,015 INFO EXTRACT from /home/alka/MLOPs-Class/spark_demo/yellow_tripdata_2023-01.parquet
2026-05-21 16:14:12,167 INFO   extracted 3,066,766 rows, 19 columns
2026-05-21 16:14:12,168 INFO   schema OK — 3 required columns present
26/05/21 16:14:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/21 16:14:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/21 16:14:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/05/21 16:14:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
2026-05-21 16:14:30,467 INFO   loaded → /home/alka/MLOPs-Class/spark_demo/etl_output
2026-05-21 16:1


Output directories (partitioned by year/month):
  year=2008/
  year=2022/
  year=2023/


In [12]:
# --- Incremental Pipeline + Idempotency ---
print("=== Incremental Pipeline: process only NEW data ===\n")

CHECKPOINT_FILE = f"{DATA_DIR}/etl_checkpoint.json"

def get_last_processed():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            return json.load(f).get("last_date")
    return None

def save_checkpoint(last_date_str):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"last_date": last_date_str, "ts": time.time()}, f)

def incremental_extract(spark, path, last_date=None):
    df = spark.read.parquet(path)
    if last_date:
        df = df.filter(F.to_date("tpep_pickup_datetime") > last_date)
        logger.info(f"Incremental: after {last_date}")
    else:
        logger.info("Full load (no checkpoint)")
    return df

# Simulate run 1 — no checkpoint yet
last = get_last_processed()
df_i = incremental_extract(spark, DATA_2023, last)
count_i = df_i.count()
max_date = df_i.agg(F.max(F.to_date("tpep_pickup_datetime"))).collect()[0][0]
save_checkpoint(str(max_date))
print(f"Run 1: processed {count_i:,} rows up to {max_date}")
print(f"Checkpoint saved: {max_date}")

# Simulate run 2 — checkpoint exists, nothing new
last2 = get_last_processed()
df_i2 = incremental_extract(spark, DATA_2023, last2)
print(f"Run 2: {df_i2.count():,} rows (0 expected — idempotent!)")

print("""
Idempotency guarantee: write with mode='overwrite' per partition.
  If the job crashes mid-write, re-running overwrites the partial output
  → output always reflects a complete run, never partial data.
""")

2026-05-21 16:14:37,237 INFO Full load (no checkpoint)


=== Incremental Pipeline: process only NEW data ===



NameError: name 'json' is not defined

---
## 17. Section 11 — Workflow Orchestration (Apache Airflow)

A standalone PySpark script has no retry, no scheduling, no dependency tracking, and no visibility. **Orchestrators** solve this.

```
script.py → broken, no retry, runs manually
Airflow DAG → scheduled, retried, monitored, with alerts
```

The cell below shows a production Airflow DAG for the taxi ETL. It is not executed here — it would run on an Airflow server. Study the structure: how tasks are declared, how dependencies flow, and how scheduling + retries are configured.

In [13]:
AIRFLOW_DAG = '''
# airflow/dags/nyc_taxi_monthly_etl.py
# ─────────────────────────────────────
# Runs on the 1st of each month at 06:00 UTC.
# Downloads the previous month's taxi file, validates, runs ETL, checks quality.

from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.sensors.filesystem import FileSensor

default_args = {
    "owner":            "data_engineering",
    "retries":          3,
    "retry_delay":      timedelta(minutes=10),
    "email_on_failure": True,
    "email":            ["data-team@company.com"],
}

with DAG(
    dag_id="nyc_taxi_monthly_etl",
    default_args=default_args,
    schedule_interval="0 6 1 * *",   # cron: 6 AM on the 1st
    start_date=datetime(2024, 1, 1),
    catchup=False,                    # don't backfill missed runs
    tags=["etl", "taxi", "monthly"],
) as dag:

    # ── Task 1: wait for upstream file ──────────────────────────────────────
    wait_for_file = FileSensor(
        task_id="wait_for_parquet",
        filepath="/data/raw/yellow_tripdata_{{ macros.ds_format(ds, '%Y-%m', '%Y-%m') }}.parquet",
        poke_interval=300,   # check every 5 minutes
        timeout=7200,        # give up after 2 hours
        mode="reschedule",   # release worker slot while waiting
    )

    # ── Task 2: validate schema ──────────────────────────────────────────────
    validate_task = BashOperator(
        task_id="validate_schema",
        bash_command=(
            "python /opt/airflow/scripts/validate_taxi.py "
            "--month={{ macros.ds_format(ds, '%Y-%m', '%Y-%m') }}"
        ),
    )

    # ── Task 3: run Spark ETL job ────────────────────────────────────────────
    etl_task = BashOperator(
        task_id="spark_etl",
        bash_command=(
            "spark-submit --master yarn --deploy-mode cluster "
            "/opt/airflow/jobs/taxi_etl.py "
            "--month={{ macros.ds_format(ds, '%Y-%m', '%Y-%m') }}"
        ),
        execution_timeout=timedelta(hours=2),
    )

    # ── Task 4: data quality checks ──────────────────────────────────────────
    dq_task = BashOperator(
        task_id="data_quality",
        bash_command=(
            "python /opt/airflow/scripts/dq_checks.py "
            "--month={{ macros.ds_format(ds, '%Y-%m', '%Y-%m') }}"
        ),
    )

    # ── Task 5: notify on success ────────────────────────────────────────────
    notify_task = BashOperator(
        task_id="notify_success",
        bash_command="python /opt/airflow/scripts/notify_slack.py --status=success",
        trigger_rule="all_success",
    )

    # ── Dependency graph (the DAG) ───────────────────────────────────────────
    #   wait → validate → etl → dq → notify
    wait_for_file >> validate_task >> etl_task >> dq_task >> notify_task
'''

print(AIRFLOW_DAG)

print("""
Key concepts:
  schedule_interval : cron expression controlling when the DAG runs
  catchup=False     : don't run past intervals when the DAG is first enabled
  retries/retry_delay: automatic retry on task failure
  FileSensor        : pause until an upstream file arrives
  trigger_rule      : control when a task fires (all_success, all_failed, one_failed…)
  {{ ds }}          : Jinja templating — Airflow injects the execution date
  >>                : dependency operator — A >> B means B runs after A
""")


# airflow/dags/nyc_taxi_monthly_etl.py
# ─────────────────────────────────────
# Runs on the 1st of each month at 06:00 UTC.
# Downloads the previous month's taxi file, validates, runs ETL, checks quality.

from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.bash import BashOperator
from airflow.sensors.filesystem import FileSensor

default_args = {
    "owner":            "data_engineering",
    "retries":          3,
    "retry_delay":      timedelta(minutes=10),
    "email_on_failure": True,
    "email":            ["data-team@company.com"],
}

with DAG(
    dag_id="nyc_taxi_monthly_etl",
    default_args=default_args,
    schedule_interval="0 6 1 * *",   # cron: 6 AM on the 1st
    start_date=datetime(2024, 1, 1),
    catchup=False,                    # don't backfill missed runs
    tags=["etl", "taxi", "monthly"],
) as dag:

    # ── Task 1: wait for upstream file ────────────────────────────

---
## 18. Section 12 — Structured Streaming

Spark Structured Streaming treats a real-time data stream as an **unbounded table**. New data appends rows; queries run continuously.

```
Stream source → [Spark Streaming Engine] → Stream sink
   (Kafka, files, sockets)                  (Parquet, Kafka, console)
```

| Concept | Description |
|---------|-------------|
| Trigger | How often Spark processes new data (micro-batch interval) |
| Output mode | `append` / `update` / `complete` |
| Watermark | How late data is tolerated before state is dropped |
| Checkpoint | Tracks progress so restarts resume without reprocessing |
| Window | Group events by event-time bucket (tumbling / sliding) |

In [ ]:
STREAM_DIR   = f"{DATA_DIR}/stream_input"
STREAM_CKPT  = f"{DATA_DIR}/stream_ckpt_agg"
STREAM_CKPT2 = f"{DATA_DIR}/stream_ckpt_flat"

for d in [STREAM_DIR, STREAM_CKPT, STREAM_CKPT2]:
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)

# Write seed file — simulates arriving streaming data
seed = (
    sdf.filter(F.year("tpep_pickup_datetime") == 2023)
       .select("tpep_pickup_datetime","fare_amount","trip_distance","PULocationID","payment_type")
       .limit(50_000)
)
seed.write.mode("overwrite").parquet(f"{STREAM_DIR}/batch_0.parquet")
print(f"Seeded {seed.count():,} rows to stream input directory")

# Explicit schema required for streaming (no inferSchema)
stream_schema = StructType([
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("fare_amount",          DoubleType(),    True),
    StructField("trip_distance",        DoubleType(),    True),
    StructField("PULocationID",         LongType(),      True),
    StructField("payment_type",         LongType(),      True),
])

# Streaming DataFrame — same API as batch
stream_df = (
    spark.readStream
    .schema(stream_schema)
    .option("maxFilesPerTrigger", 1)
    .parquet(STREAM_DIR)
)
print(f"isStreaming: {stream_df.isStreaming}")

# Transform — identical to batch DataFrame transformations
stream_clean = (
    stream_df
    .filter(F.col("fare_amount") > 0)
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("fare_band",
        F.when(F.col("fare_amount") < 10, "low")
         .when(F.col("fare_amount") < 25, "medium")
         .otherwise("high")
    )
)

# Windowed aggregation: 1-hour tumbling window on event time, with watermark
windowed = (
    stream_clean
    .withWatermark("tpep_pickup_datetime", "10 minutes")  # drop state for windows > 10 min late
    .groupBy(
        F.window("tpep_pickup_datetime", "1 hour"),        # tumbling window
        "fare_band"
    )
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.sum("fare_amount"), 0).alias("revenue"),
    )
)

# Write to in-memory table (use parquet/kafka in production)
q = (
    windowed.writeStream
    .outputMode("update")
    .format("memory")
    .queryName("taxi_windows")
    .option("checkpointLocation", STREAM_CKPT)
    .trigger(processingTime="0 seconds")
    .start()
)
q.awaitTermination(timeout=30)
print(f"\nStream status: {q.status['message']}")

In [ ]:
# Inspect windowed aggregation results
print("=== 1-Hour Tumbling Windows (event time) ===")
spark.sql("SELECT window.start, window.end, fare_band, trips, avg_fare, revenue FROM taxi_windows ORDER BY window.start, fare_band").show(20, truncate=False)
q.stop()

# Simulate new data arriving (second micro-batch)
print("\n=== Simulating second micro-batch (new file arrives) ===")
sdf.filter(F.year("tpep_pickup_datetime") == 2023).select(
    "tpep_pickup_datetime","fare_amount","trip_distance","PULocationID","payment_type"
).limit(10_000).write.mode("overwrite").parquet(f"{STREAM_DIR}/batch_1.parquet")
print("New file written to stream input directory.")

print("""
Output modes explained:
  append   → only NEW rows from this trigger (no updates to past rows)
             use for: simple stateless transformations
  update   → only CHANGED rows (for aggregations that get updated)
             use for: running totals, windowed aggregations
  complete → FULL result table every trigger
             use for: small aggregations shown on a dashboard

Watermark:
  .withWatermark("event_time", "10 minutes")
  Tells Spark: if an event arrives more than 10 min late, discard it.
  Without watermark: Spark keeps ALL state forever → OOM risk.
""")

---
## 19. Section 13 — Data Lakes and Lakehouse Concepts

```
Data Warehouse    : structured, schema-on-write, fast queries, expensive storage
Data Lake         : raw files, schema-on-read, cheap storage, risk of "data swamp"
Lakehouse         : combines both — open formats (Parquet/Delta) + ACID + SQL
```

**Medallion Architecture** (Bronze → Silver → Gold):

```
Raw source files
     │
  Bronze  ← raw ingest, minimal transformation, append-only, full history
     │
  Silver  ← cleaned, validated, deduplicated, typed, enriched
     │
  Gold    ← business aggregates, ready for BI/ML, heavily optimized
```

**Delta Lake** adds ACID transactions, time travel, and schema enforcement on top of Parquet. We simulate its key concepts using partitioned Parquet (no extra dependencies needed).

In [ ]:
BRONZE = f"{DATA_DIR}/medallion/bronze"
SILVER = f"{DATA_DIR}/medallion/silver"
GOLD   = f"{DATA_DIR}/medallion/gold"

for p in [BRONZE, SILVER, GOLD]:
    if os.path.exists(p): shutil.rmtree(p)
    os.makedirs(p)

# ── BRONZE: raw ingest ─────────────────────────────────────────────────────────
# Rule: never delete, never modify raw data. Add metadata only.
print("=== BRONZE: Raw Ingest ===")
t0 = time.perf_counter()
bronze_df = (
    sdf
    .withColumn("_ingested_at",  F.current_timestamp())
    .withColumn("_source_file",  F.input_file_name())
    .withColumn("_year",         F.year("tpep_pickup_datetime"))
    .withColumn("_month",        F.month("tpep_pickup_datetime"))
)
bronze_df.write.mode("overwrite").partitionBy("_year","_month").parquet(BRONZE)
print(f"  {bronze_df.count():,} rows written in {time.perf_counter()-t0:.2f}s")
print(f"  Partitions: {[d for d in sorted(os.listdir(BRONZE)) if not d.startswith('.')[:5]]}")

# ── SILVER: clean and validate ────────────────────────────────────────────────
print("\n=== SILVER: Clean, Validate, Enrich ===")
t0 = time.perf_counter()
raw_df = spark.read.parquet(BRONZE)

silver_df = (
    raw_df
    .filter(F.col("fare_amount").between(0.5, 1000))
    .filter(F.col("trip_distance").between(0, 500))
    .filter(F.col("passenger_count").between(1, 9))
    .filter(F.col("tpep_pickup_datetime") < F.col("tpep_dropoff_datetime"))
    .fillna({"airport_fee": 0.0, "congestion_surcharge": 0.0})
    .dropDuplicates(["tpep_pickup_datetime","PULocationID","DOLocationID","fare_amount"])
    .withColumn("duration_min", F.round(
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60, 1
    ))
    .withColumn("speed_mph",  F.round(F.col("trip_distance") / (F.col("duration_min")/60 + 0.001), 1))
    .withColumn("is_airport", F.col("RatecodeID").isin([2,3]).cast("boolean"))
    .withColumn("_silver_at", F.current_timestamp())
)
silver_df.write.mode("overwrite").partitionBy("_year","_month").parquet(SILVER)
bronze_n = raw_df.count(); silver_n = silver_df.count()
print(f"  Bronze: {bronze_n:,} → Silver: {silver_n:,}  (removed {bronze_n-silver_n:,} invalid, {(bronze_n-silver_n)/bronze_n*100:.1f}%)")
print(f"  Written in {time.perf_counter()-t0:.2f}s")

In [ ]:
# ── GOLD: business aggregates ─────────────────────────────────────────────────
print("=== GOLD: Business-Ready Aggregates ===")
t0 = time.perf_counter()
sv = spark.read.parquet(SILVER)

# Gold table 1: daily KPIs
gold_daily = (
    sv.groupBy(F.to_date("tpep_pickup_datetime").alias("date"), "_year","_month")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.sum("fare_amount"), 0).alias("revenue"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_dist_mi"),
        F.round(F.avg("duration_min"), 1).alias("avg_dur_min"),
        F.round(F.sum("tip_amount") / F.sum("fare_amount") * 100, 2).alias("tip_rate_pct"),
        F.countDistinct("PULocationID").alias("active_zones"),
    )
    .orderBy("date")
)
gold_daily.write.mode("overwrite").partitionBy("_year","_month").parquet(f"{GOLD}/daily_kpis")

# Gold table 2: zone performance (for BI / mapping)
gold_zones = (
    sv.groupBy("PULocationID","_year")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.sum("fare_amount"), 0).alias("total_revenue"),
        F.round(F.avg("speed_mph"), 1).alias("avg_speed_mph"),
    )
    .orderBy(F.col("trips").desc())
)
gold_zones.write.mode("overwrite").partitionBy("_year").parquet(f"{GOLD}/zone_performance")

print(f"  Written in {time.perf_counter()-t0:.2f}s")
print("\nSample daily KPIs (Gold):")
gold_daily.filter(F.col("_year") == 2023).show(5)

print("Top-10 pickup zones by revenue (2023):")
gold_zones.filter(F.col("_year") == 2023).show(10)

# ── Delta Lake concepts (simulated with versioned Parquet) ────────────────────
print("=== Delta Lake Concepts ===")
print("""
Delta Lake (open source) adds to Parquet:
  ACID transactions : concurrent reads/writes without corruption
  Time travel       : query historical snapshots
                      spark.read.format("delta").option("versionAsOf", 3).load(path)
  Schema evolution  : add columns without breaking existing readers
  MERGE (upsert)    : update-or-insert in one atomic operation
  Vacuum            : clean up old file versions to save storage

Simulating time travel with versioned Parquet directories:
"""
)
# Version 1: 2022 data
v1_path = f"{DATA_DIR}/versioned/v1"
if os.path.exists(v1_path): shutil.rmtree(v1_path)
spark.read.parquet(DATA_2022).write.mode("overwrite").parquet(v1_path)
print(f"v1 (Jan 2022): {spark.read.parquet(v1_path).count():,} rows")

# Version 2: add 2023 data
v2_path = f"{DATA_DIR}/versioned/v2"
if os.path.exists(v2_path): shutil.rmtree(v2_path)
spark.read.parquet(DATA_2022, DATA_2023).write.mode("overwrite").parquet(v2_path)
print(f"v2 (Jan 2022+2023): {spark.read.parquet(v2_path).count():,} rows")

# "Time travel": read v1 (yesterday's snapshot)
print(f"Read v1 snapshot (time travel equivalent): {spark.read.parquet(v1_path).count():,} rows")

---
## 20. Section 14 — Performance Optimization

| Technique | When to use | Impact |
|-----------|------------|--------|
| `cache()` / `persist()` | DataFrame used 3+ times | Avoid recomputation |
| `explain()` | Debugging slow queries | Understand what Spark actually does |
| Broadcast join | One side < ~100 MB | Eliminates shuffle on large side |
| Partition tuning | After a shuffle / before write | Match parallelism to data size |
| Column pruning | Parquet queries | Read only needed columns |
| Avoid `collect()` | Always, except small results | Prevents OOM on driver |
| Avoid UDFs | Use built-in functions instead | Stays in JVM, no serialization |

In [ ]:
from pyspark import StorageLevel

print("=== Caching and Persistence ===\n")

# Expensive computation — filter + enrich
def make_expensive_df():
    return (
        sv.filter(F.col("fare_amount") > 5)
          .withColumn("speed_bin",
              F.when(F.col("speed_mph") < 10, "slow")
               .when(F.col("speed_mph") < 30, "normal")
               .otherwise("fast"))
    )

# Without cache: recomputes from scratch for every action
t0 = time.perf_counter()
df_nc = make_expensive_df()
_ = df_nc.count()
_ = df_nc.agg(F.avg("fare_amount")).collect()
_ = df_nc.groupBy("speed_bin").count().collect()
t_no_cache = time.perf_counter() - t0
print(f"Without cache (3 actions): {t_no_cache:.2f}s")

# With cache: compute once, reuse in memory
t0 = time.perf_counter()
df_cached = make_expensive_df().cache()
_ = df_cached.count()           # triggers caching
_ = df_cached.agg(F.avg("fare_amount")).collect()
_ = df_cached.groupBy("speed_bin").count().collect()
t_cache = time.perf_counter() - t0
df_cached.unpersist()
print(f"With cache     (3 actions): {t_cache:.2f}s  (speedup: {t_no_cache/max(t_cache,0.001):.1f}×)")

# StorageLevel options
print("""
StorageLevel options:
  MEMORY_ONLY         → default for cache() — fastest, fails if OOM
  MEMORY_AND_DISK     → spills to disk if memory full — safer
  DISK_ONLY           → always on disk — slowest but no OOM
  MEMORY_ONLY_SER     → compressed in memory — less RAM, slower access
  MEMORY_AND_DISK_2   → 2 replicas — fault tolerant in cluster mode
""")
# Example with explicit storage level
df_persisted = make_expensive_df().persist(StorageLevel.MEMORY_AND_DISK)
_ = df_persisted.count()
print(f"MEMORY_AND_DISK cache size: {spark.sparkContext._jvm.org.apache.spark.storage.StorageUtils.memoryUsed():,} bytes in JVM storage")
df_persisted.unpersist()

In [ ]:
print("=== Explain Plans: Reading Spark's Physical Plan ===\n")

q_explain = (
    sv.filter(F.col("_year") == 2023)
    .filter(F.col("fare_amount") > 10)
    .groupBy("PULocationID")
    .agg(F.sum("fare_amount").alias("revenue"), F.count("*").alias("trips"))
    .orderBy(F.col("revenue").desc())
    .limit(10)
)
q_explain.explain(mode="formatted")

print("=== Key plan nodes to recognize ===")
print("""
FileScan parquet ... PushedFilters: [IsNotNull(fare_amount), GreaterThan(fare_amount,10.0)]
  → Predicate pushdown active: Parquet skips row groups that don't match

PartitionFilters: [isnotnull(_year#123), (_year#123 = 2023)]
  → Partition pruning: only reads the _year=2023 directory

Exchange hashpartitioning(PULocationID, 8)
  → A shuffle is happening (expensive!) — all rows with same zone go to same partition

BroadcastHashJoin BuildRight
  → Small table was broadcast — no shuffle on the large side

Sort [revenue DESC], true, 0
  → Global sort required — unavoidable for ORDER BY with LIMIT

PartialAgg → FinalAgg
  → Two-phase aggregation: partial sums in map phase, final sums after shuffle
""")

print("=== Partition Tuning ===")
spark.conf.set("spark.sql.shuffle.partitions", "8")
t0 = time.perf_counter()
_ = sv.groupBy("PULocationID").agg(F.sum("fare_amount")).collect()
t8 = time.perf_counter() - t0

spark.conf.set("spark.sql.shuffle.partitions", "200")
t0 = time.perf_counter()
_ = sv.groupBy("PULocationID").agg(F.sum("fare_amount")).collect()
t200 = time.perf_counter() - t0

spark.conf.set("spark.sql.shuffle.partitions", "8")
print(f"8 partitions  : {t8:.2f}s   ← right-sized for small data")
print(f"200 partitions: {t200:.2f}s  ← over-partitioned, scheduler overhead dominates")
print("\nRule of thumb: shuffle_partitions ≈ max(8, data_GB × 20). Default 200 is for large clusters.")

In [ ]:
print("=== Avoiding collect() — most important safety rule ===\n")

print("""
NEVER:  df.collect()           # pulls ALL data to driver → OOM if df is large
NEVER:  df.toPandas()          # same issue — entire DataFrame to driver
NEVER:  list(df.toLocalIterator())  # streams but still driver-bound

SAFE alternatives:
  df.show(n)               # display first n rows (does not pull all data)
  df.limit(n).collect()    # bounded — safe for any DataFrame size
  df.write.parquet(path)   # write to storage without collecting
  df.count()               # single integer — fine
  df.agg(...).collect()    # aggregate first, then collect (tiny result)
  df.sample(0.01).toPandas()  # 1% sample for exploration
""")

# Demonstrate safe patterns
print("Safe: count")
print(f"  sv.count() = {sv.count():,}")

print("\nSafe: limit().collect()")
sample = sv.select("fare_amount","trip_distance").limit(5).collect()
print(f"  first 5 rows: {[r['fare_amount'] for r in sample]}")

print("\nSafe: aggregate then collect")
stats = sv.agg(F.min("fare_amount"), F.max("fare_amount"), F.avg("fare_amount")).collect()
print(f"  min={stats[0][0]:.2f}  max={stats[0][1]:.2f}  avg={stats[0][2]:.2f}")

print("\nSafe: write to Parquet (no driver memory used for data)")
out = f"{DATA_DIR}/safe_write_test"
if os.path.exists(out): shutil.rmtree(out)
sv.filter(F.col("_year") == 2023).coalesce(1).write.mode("overwrite").parquet(out)
print(f"  Written to {out}")

---
## 21. Section 15 — Data Quality and Reliability

Bad data costs more to fix downstream than to catch at ingestion. Data quality is not optional in production pipelines.

| Check type | Example |
|-----------|---------|
| Completeness | No null in required columns |
| Validity | fare_amount ∈ [0.5, 1000] |
| Consistency | pickup_time < dropoff_time |
| Uniqueness | No exact duplicate trips |
| Timeliness | Pickup date within expected range |
| Business rules | No fare > $100 for < 0.5 mi trips |

In [ ]:
def run_dq_checks(df, dataset_name):
    """Run a suite of data quality checks. Returns list of result dicts."""
    results = []
    total = df.count()
    if total == 0:
        print(f"[DQ] {dataset_name}: 0 rows — skip")
        return results

    def check(name, condition_col, description):
        failed = df.filter(~condition_col).count()
        pct = failed / total * 100
        status = "PASS" if failed == 0 else ("WARN" if pct < 1.0 else "FAIL")
        results.append({
            "check": name, "status": status,
            "failed_rows": failed, "pct_failed": round(pct, 3),
            "description": description,
        })

    # Completeness
    check("fare_not_null",     F.col("fare_amount").isNotNull(),            "fare_amount must not be null")
    check("pickup_not_null",   F.col("tpep_pickup_datetime").isNotNull(),   "pickup datetime must not be null")
    check("distance_not_null", F.col("trip_distance").isNotNull(),          "trip_distance must not be null")

    # Validity
    check("fare_positive",     F.col("fare_amount") > 0,                    "fare must be > 0")
    check("fare_range",        F.col("fare_amount").between(0.5, 1000),     "fare must be $0.50 – $1000")
    check("distance_non_neg",  F.col("trip_distance") >= 0,                 "distance must be >= 0")
    check("payment_valid",     F.col("payment_type").isin([1,2,3,4,5,6]),   "payment type must be 1–6")

    # Consistency (temporal)
    check("pickup_before_dropoff",
          F.col("tpep_pickup_datetime") < F.col("tpep_dropoff_datetime"),
          "pickup must be before dropoff")
    check("reasonable_duration",
          (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")).between(60, 7200),
          "trip duration must be 1 min – 2 hours")

    # Business rule
    check("no_high_fare_short_trip",
          ~((F.col("fare_amount") > 100) & (F.col("trip_distance") < 0.5)),
          "fare > $100 for < 0.5 mi is anomalous")

    report_df = spark.createDataFrame(results)
    print(f"\n{'='*65}")
    print(f"DQ Report: {dataset_name}  ({total:,} rows, {len(results)} checks)")
    print(f"{'='*65}")
    report_df.orderBy("status","check").show(truncate=False)

    n_pass = sum(1 for r in results if r["status"] == "PASS")
    n_warn = sum(1 for r in results if r["status"] == "WARN")
    n_fail = sum(1 for r in results if r["status"] == "FAIL")
    print(f"Summary: {n_pass} PASS  {n_warn} WARN  {n_fail} FAIL")
    return results

# Run on silver layer (already cleaned — most checks should pass)
dq_results = run_dq_checks(sv.filter(F.col("_year") == 2023), "NYC Taxi Silver 2023")

In [ ]:
print("=== Duplicate Detection ===\n")
total = sv.count()
exact_distinct = sv.distinct().count()
print(f"Total rows    : {total:,}")
print(f"Distinct rows : {exact_distinct:,}")
print(f"Exact dupes   : {total - exact_distinct:,}")

# Near-duplicates: same business-key trip but different fare
business_key = ["tpep_pickup_datetime","PULocationID","DOLocationID"]
w_dedup = Window.partitionBy(*business_key).orderBy(F.col("fare_amount").desc())
df_deduped = sv.withColumn("rn", F.row_number().over(w_dedup)).filter(F.col("rn") == 1).drop("rn")
print(f"After near-dedup on {business_key}: {df_deduped.count():,} rows")

print("\n=== Schema Enforcement ===\n")

def enforce_schema(df, required_fields):
    """Check df matches expected column types. Returns True if OK."""
    ok = True
    for name, expected_type in required_fields.items():
        if name not in df.columns:
            print(f"  [MISSING]     {name}")
            ok = False
        else:
            actual = str(df.schema[name].dataType)
            if actual != expected_type:
                print(f"  [TYPE MISMATCH] {name}: expected {expected_type}, got {actual}")
                ok = False
            else:
                print(f"  [OK]          {name}: {actual}")
    return ok

required = {
    "fare_amount":          "DoubleType()",
    "tpep_pickup_datetime": "TimestampNTZType()",
    "trip_distance":        "DoubleType()",
    "PULocationID":         "LongType()",
    "nonexistent_col":      "StringType()",   # will flag MISSING
}
schema_ok = enforce_schema(sv, required)
print(f"\nSchema validation: {'PASS' if schema_ok else 'FAIL (see above)'}")

In [ ]:
print("=== Unit Testing PySpark Pipelines ===\n")
print("Test with small in-memory DataFrames — fast and deterministic.\n")

def test_transform_valid_row():
    """transform() keeps rows that pass all filters."""
    good = spark.createDataFrame([(
        1, "2023-01-15 10:00:00", "2023-01-15 10:20:00",
        1.0, 5.0, 1.0, "N", 161, 236, 1,
        15.0, 0.5, 0.5, 3.0, 0.0, 0.3, 19.3, 2.5, 0.0
    )], sdf.schema)
    result = transform(good)
    assert result.count() == 1, "Valid row should pass transform"
    row = result.first()
    assert row["fare_band"] == "medium", f"Expected medium, got {row['fare_band']}"
    assert row["duration_min"] == 20.0, f"Expected 20.0 min, got {row['duration_min']}"
    print("  PASS: test_transform_valid_row")

def test_transform_filters_negative_fare():
    """transform() must drop rows with negative fare."""
    bad = spark.createDataFrame([(
        1, "2023-01-15 11:00:00", "2023-01-15 11:30:00",
        1.0, 2.0, 1.0, "N", 237, 142, 2,
        -5.0, 0.0, 0.5, 0.0, 0.0, 0.3, -4.2, 0.0, 0.0
    )], sdf.schema)
    assert transform(bad).count() == 0, "Negative fare must be filtered"
    print("  PASS: test_transform_filters_negative_fare")

def test_transform_filters_null_pickup():
    """transform() must drop rows with null pickup time."""
    null_pickup = spark.createDataFrame([(
        1, None, "2023-01-15 12:30:00",
        2.0, 2.0, 1.0, "N", 170, 230, 1,
        12.0, 0.5, 0.5, 2.0, 0.0, 0.3, 15.3, 2.5, 0.0
    )], sdf.schema)
    assert transform(null_pickup).count() == 0, "Null pickup must be filtered"
    print("  PASS: test_transform_filters_null_pickup")

def test_dq_catches_bad_data():
    """run_dq_checks must report FAILs for known-bad data."""
    bad = spark.createDataFrame([
        (1, "2023-01-01 10:00:00", "2023-01-01 09:00:00", 1.0, 5.0, 1.0, "N", 161, 236, 1,
         15.0, 0.5, 0.5, 3.0, 0.0, 0.3, 19.3, 2.5, 0.0),   # dropoff before pickup
        (2, "2023-01-01 11:00:00", "2023-01-01 11:30:00", 1.0, 0.3, 1.0, "N", 237, 142, 2,
         -5.0, 0.0, 0.5, 0.0, 0.0, 0.3, -4.2, 0.0, 0.0),   # negative fare
    ], sdf.schema)
    results = run_dq_checks(bad, "test_bad_data")
    fails = [r for r in results if r["status"] == "FAIL"]
    assert len(fails) >= 2, f"Expected ≥2 FAIL, got {fails}"
    print("  PASS: test_dq_catches_bad_data")

# Run all tests
for test_fn in [test_transform_valid_row, test_transform_filters_negative_fare,
                test_transform_filters_null_pickup, test_dq_catches_bad_data]:
    try:
        test_fn()
    except AssertionError as e:
        print(f"  FAIL: {test_fn.__name__} — {e}")

print("\nAll pipeline tests complete.")

---
## 22. Complete Course Summary

| Section | Key takeaway |
|---------|-------------|
| **6 DataFrame API** | Use built-in functions, explicit schemas, and window functions. Avoid regular UDFs — use Pandas UDFs when custom logic is unavoidable. |
| **7 Spark SQL** | SQL and DataFrame API produce identical plans. Use CTEs for readable multi-step queries. `explain()` shows what Spark will actually do. |
| **8 Distributed Processing** | Partitions = parallelism. Shuffle = expensive. Coalesce to reduce, repartition to balance. Broadcast small tables to eliminate shuffle. |
| **9 Storage Formats** | Parquet > ORC > JSON > CSV for analytics. Columnar + compression + statistics = fast scans. Default to Parquet + Snappy or ZStd. |
| **10 ETL Design** | Extract → validate → transform → load. Partition output by date. Idempotent writes with `mode=overwrite`. Checkpoints for incremental loads. |
| **11 Orchestration** | Airflow DAGs replace cron jobs. Tasks declare dependencies with `>>`. Use sensors to wait for upstream data. Retries are automatic. |
| **12 Streaming** | Same DataFrame API works for batch and streaming. Watermarks bound memory. Choose output mode based on statefulness. |
| **13 Lakehouse** | Bronze = raw + metadata. Silver = clean + typed + enriched. Gold = aggregated business tables. Delta Lake adds ACID + time travel. |
| **14 Performance** | Cache reused DataFrames. Tune shuffle partitions. Read explain plans. Broadcast lookup tables. Never `collect()` large DataFrames. |
| **15 Data Quality** | Validate at every layer boundary. Report completeness, validity, consistency, uniqueness, business rules. Unit-test with in-memory DataFrames. |

In [ ]:
# Clean up temp directories created during this session
for d in ["stream_input","stream_ckpt_agg","stream_ckpt_flat","safe_write_test"]:
    p = os.path.join(DATA_DIR, d)
    if os.path.exists(p): shutil.rmtree(p)

spark.stop()
print("SparkSession stopped. Notebook complete.")